In [1]:

import sys
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "02_processed_data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "01_src"
DATA_ROOT = PROJECT_ROOT / "02_processed_data"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from config.pipeline_config import (
    TRAINING_SIZE,
    VALIDATION_SIZE,
    TEST_SIZE,
    CAPTIONS_PER_IMAGE,
    get_pipeline_snapshot,
    check_pipeline_compatibility,
)

from embeddings.clip_image_embeddings import (
    generate_clip_image_embeddings
)

print("Project root:", PROJECT_ROOT)
print("Source root:", SRC_ROOT)
print("Data root:", DATA_ROOT)


Project root: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG
Source root: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\01_src
Data root: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data


In [2]:
import importlib
import embeddings.clip_image_embeddings as clip_module

importlib.reload(clip_module)

print("Module reloaded successfully.")

Module reloaded successfully.


In [3]:
# ==================================================
# PREPARE A SMALL SAMPLE FOR CLIP TESTING
# ==================================================

# Load the training image records
captions_path = (
    DATA_ROOT / "training" / "annotations" / "captions.json"
)

with open(captions_path, "r", encoding="utf-8") as f:
    training_data = json.load(f)

# Select five images for the test
sample_records = sorted(
    training_data["images"],
    key=lambda record: record["id"]
)[:5]

print("Sample images selected:", len(sample_records))

for record in sample_records:
    print(record["id"], record["file_name"])

Sample images selected: 5
149 000000000149.jpg
257 000000000257.jpg
322 000000000322.jpg
397 000000000397.jpg
486 000000000486.jpg


In [4]:
# ==================================================
# DEFINE IMAGE DIRECTORY FOR SAMPLE GENERATION
# ==================================================

IMAGE_DIR = DATA_ROOT / "training" / "images"

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"

print("Image directory:", IMAGE_DIR)
print("Directory exists:", IMAGE_DIR.exists())

Image directory: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\images
Directory exists: True


In [5]:
sample_output_dir = (
    DATA_ROOT / "embedding_experiments" / "clip_sample"
)

In [6]:
sample_summary = clip_module.generate_clip_image_embeddings(
    image_records=sample_records,
    image_dir=IMAGE_DIR,
    output_dir=sample_output_dir,
    batch_size=4
)

print(json.dumps(sample_summary, indent=2))

Loading CLIP model: openai/clip-vit-base-patch32
Device: cpu


Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'openai/clip-vit-base-patch32'.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating CLIP image embeddings:   0%|          | 0/2 [00:00<?, ?it/s]


CLIP embedding generation completed.
Shape: (5, 512)
Data type: float32
Embeddings saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\clip_sample\clip_image_embeddings.npy
Mapping saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\clip_sample\image_embedding_index.json
{
  "model_name": "openai/clip-vit-base-patch32",
  "number_of_images": 5,
  "embedding_dimension": 512,
  "embedding_shape": [
    5,
    512
  ],
  "embedding_dtype": "float32",
  "embedding_norm": "L2",
  "normalized": true,
  "device": "cpu",
  "batch_size": 4,
  "embeddings_file": "c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\clip_sample\\clip_image_embeddings.npy",
  "mapping_file": "c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\clip_sample\\image_embedding_index.json",
  "status": "succ

In [7]:
sample_embeddings = np.load(
    sample_output_dir / "clip_image_embeddings.npy"
)

with open(
    sample_output_dir / "image_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    sample_mapping = json.load(f)

expected_count = len(sample_records)

assert sample_embeddings.shape == (expected_count, 512)
assert sample_embeddings.dtype == np.float32
assert np.isfinite(sample_embeddings).all()
assert len(sample_mapping) == expected_count
assert [item["row"] for item in sample_mapping] == list(
    range(expected_count)
)

assert [item["image_id"] for item in sample_mapping] == [
    record["id"] for record in sample_records
]

print("Sample embedding validation: PASSED")
print("Shape:", sample_embeddings.shape)
print("Data type:", sample_embeddings.dtype)
print("Mapping records:", len(sample_mapping))

Sample embedding validation: PASSED
Shape: (5, 512)
Data type: float32
Mapping records: 5


In [8]:
# ==================================================
# VERIFY PROJECT AND DATASET PATHS
# ==================================================

RAW_ROOT = PROJECT_ROOT / "00_original_data"
PROCESSED_ROOT = DATA_ROOT

TRAIN_IMAGES = RAW_ROOT / "train2017"

print("Project root exists:", PROJECT_ROOT.exists())
print("Raw data exists:", RAW_ROOT.exists())
print("Processed data exists:", PROCESSED_ROOT.exists())
print("Training images exist:", TRAIN_IMAGES.exists())

Project root exists: True
Raw data exists: True
Processed data exists: True
Training images exist: True


In [9]:
print("Training captions files:")
for path in (PROCESSED_ROOT / "training" / "captions").glob("*"):
    print(path.name)

print("\nTraining metadata files:")
for path in (PROCESSED_ROOT / "training" / "metadata").glob("*"):
    print(path.name)

Training captions files:
captions_cleaned.json

Training metadata files:
caption_metadata.json
image_metadata.json
preprocessing_metadata.json


In [11]:
# ==================================================
# LOAD TRAINING IMAGE RECORDS
# ==================================================

captions_path = (
    PROCESSED_ROOT / "training" / "annotations" / "captions.json"
)

with open(captions_path, "r", encoding="utf-8") as f:
    training_data = json.load(f)

training_records = sorted(
    training_data["images"],
    key=lambda record: record["id"]
)

print("Training image records:", len(training_records))
print("First 10 records selected for testing:")

for record in training_records[:10]:
    print(record["id"], record["file_name"])

Training image records: 9900
First 10 records selected for testing:
149 000000000149.jpg
257 000000000257.jpg
322 000000000322.jpg
397 000000000397.jpg
486 000000000486.jpg
562 000000000562.jpg
625 000000000625.jpg
629 000000000629.jpg
650 000000000650.jpg
690 000000000690.jpg


In [12]:
normalized_sample_dir = (
    PROCESSED_ROOT / "embedding_experiments" / "clip_normalized_sample"
)

normalized_sample_summary = clip_module.generate_clip_image_embeddings(
    image_records=training_records[:10],
    image_dir=TRAIN_IMAGES,
    output_dir=normalized_sample_dir,
    batch_size=4
)

normalized_sample_summary

Loading CLIP model: openai/clip-vit-base-patch32
Device: cpu


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating CLIP image embeddings:   0%|          | 0/3 [00:00<?, ?it/s]


CLIP embedding generation completed.
Shape: (10, 512)
Data type: float32
Embeddings saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\clip_normalized_sample\clip_image_embeddings.npy
Mapping saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\clip_normalized_sample\image_embedding_index.json


{'model_name': 'openai/clip-vit-base-patch32',
 'number_of_images': 10,
 'embedding_dimension': 512,
 'embedding_shape': [10, 512],
 'embedding_dtype': 'float32',
 'embedding_norm': 'L2',
 'normalized': True,
 'device': 'cpu',
 'batch_size': 4,
 'embeddings_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\clip_normalized_sample\\clip_image_embeddings.npy',
 'mapping_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\clip_normalized_sample\\image_embedding_index.json',
 'status': 'success'}

In [13]:
normalized_sample_embeddings = np.load(
    normalized_sample_dir / "clip_image_embeddings.npy"
)

# Calculate the L2 norm of every embedding
sample_norms = np.linalg.norm(
    normalized_sample_embeddings,
    ord=2,
    axis=1
)

print("Embedding shape:", normalized_sample_embeddings.shape)
print("Minimum L2 norm:", sample_norms.min())
print("Maximum L2 norm:", sample_norms.max())
print("Mean L2 norm:", sample_norms.mean())

# Every vector should have a norm approximately equal to 1
assert np.allclose(sample_norms, 1.0, atol=1e-5)

print("L2 normalization validation: PASSED")

Embedding shape: (10, 512)
Minimum L2 norm: 0.9999999
Maximum L2 norm: 1.0
Mean L2 norm: 1.0
L2 normalization validation: PASSED


In [14]:
import json

captions_file = (
    PROCESSED_ROOT / "training" / "captions" / "captions_cleaned.json"
)

image_metadata_file = (
    PROCESSED_ROOT / "training" / "metadata" / "image_metadata.json"
)

with open(captions_file, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

with open(image_metadata_file, "r", encoding="utf-8") as f:
    image_metadata = json.load(f)

print("CAPTIONS FILE")
print("Type:", type(captions_data).__name__)

if isinstance(captions_data, dict):
    print("Keys:", list(captions_data.keys()))
    for key, value in captions_data.items():
        print(f"\nExample from '{key}':")
        print(str(value[:1] if isinstance(value, list) else value)[:1000])
        break
elif isinstance(captions_data, list):
    print("Number of records:", len(captions_data))
    print("First record:", captions_data[0])

print("\nIMAGE METADATA FILE")
print("Type:", type(image_metadata).__name__)

if isinstance(image_metadata, dict):
    print("Keys:", list(image_metadata.keys()))
    for key, value in image_metadata.items():
        print(f"\nExample from '{key}':")
        print(str(value[:1] if isinstance(value, list) else value)[:1000])
        break
elif isinstance(image_metadata, list):
    print("Number of records:", len(image_metadata))
    print("First record:", image_metadata[0])

CAPTIONS FILE
Type: dict
Keys: ['info', 'licenses', 'images', 'annotations']

Example from 'info':
{'description': 'COCO 2017 Dataset', 'url': 'http://cocodataset.org', 'version': '1.0', 'year': 2017, 'contributor': 'COCO Consortium', 'date_created': '2017/09/01'}

IMAGE METADATA FILE
Type: dict
Keys: ['split', 'image_count', 'images']

Example from 'split':
training


In [15]:
print("CAPTIONS FILE")
print("Number of images:", len(captions_data["images"]))
print("First image record:")
print(captions_data["images"][0])

print("\nIMAGE METADATA FILE")
print("Number of images:", len(image_metadata["images"]))
print("First image record:")
print(image_metadata["images"][0])

CAPTIONS FILE
Number of images: 9900
First image record:
{'license': 3, 'file_name': '000000184613.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000184613.jpg', 'height': 336, 'width': 500, 'date_captured': '2013-11-14 12:36:29', 'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg', 'id': 184613}

IMAGE METADATA FILE
Number of images: 9900
First image record:
{'image_id': 184613, 'file_name': '000000184613.jpg', 'original_width': 500, 'original_height': 336, 'original_mode': 'RGB', 'format': 'JPEG', 'file_size_bytes': 182585, 'original_exif_orientation': 1}


In [16]:
training_records = captions_data["images"]

print("Training records:", len(training_records))
print("First record:", training_records[0])

Training records: 9900
First record: {'license': 3, 'file_name': '000000184613.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000184613.jpg', 'height': 336, 'width': 500, 'date_captured': '2013-11-14 12:36:29', 'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg', 'id': 184613}


In [17]:
# Check the number of records
assert len(training_records) == TRAINING_SIZE

# Check unique image IDs
training_ids = [record["id"] for record in training_records]
assert len(set(training_ids)) == TRAINING_SIZE

# Check unique filenames
training_filenames = [record["file_name"] for record in training_records]
assert len(set(training_filenames)) == TRAINING_SIZE

# Check that every image exists
missing_images = [
    filename
    for filename in training_filenames
    if not (TRAIN_IMAGES / filename).exists()
]

assert len(missing_images) == 0, (
    f"Missing images: {len(missing_images)}"
)

print("Training records validation: PASSED")
print("Unique image IDs:", len(set(training_ids)))
print("Unique filenames:", len(set(training_filenames)))
print("Missing images:", len(missing_images))

Training records validation: PASSED
Unique image IDs: 9900
Unique filenames: 9900
Missing images: 0


In [55]:
training_embedding_dir = (
    PROCESSED_ROOT / "training" / "embeddings"
)

training_summary = clip_module.generate_clip_image_embeddings(
    image_records=training_records,
    image_dir=TRAIN_IMAGES,
    output_dir=training_embedding_dir,
    batch_size=16
)

training_summary

Loading CLIP model: openai/clip-vit-base-patch32
Device: cpu


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating CLIP image embeddings:   0%|          | 0/619 [00:00<?, ?it/s]


CLIP embedding generation completed.
Shape: (9900, 512)
Data type: float32
Embeddings saved: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings\clip_image_embeddings.npy
Mapping saved: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings\image_embedding_index.json


{'model_name': 'openai/clip-vit-base-patch32',
 'number_of_images': 9900,
 'embedding_dimension': 512,
 'embedding_shape': [9900, 512],
 'embedding_dtype': 'float32',
 'embedding_norm': 'L2',
 'normalized': True,
 'device': 'cpu',
 'batch_size': 16,
 'embeddings_file': 'C:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\training\\embeddings\\clip_image_embeddings.npy',
 'mapping_file': 'C:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\training\\embeddings\\image_embedding_index.json',
 'status': 'success'}

In [19]:
# Restore the training embedding directory

training_embedding_dir = (
    PROCESSED_ROOT / "training" / "embeddings"
)

assert training_embedding_dir.exists(), (
    f"Training embedding directory not found: {training_embedding_dir}"
)

print("Training embedding directory:", training_embedding_dir)
print("Directory exists:", training_embedding_dir.exists())

Training embedding directory: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings
Directory exists: True


In [20]:
import numpy as np
import json

training_embeddings_file = (
    training_embedding_dir / "clip_image_embeddings.npy"
)

training_mapping_file = (
    training_embedding_dir / "image_embedding_index.json"
)

# Reload the saved embeddings
training_embeddings = np.load(training_embeddings_file)

# Reload the mapping
with open(training_mapping_file, "r", encoding="utf-8") as f:
    training_mapping = json.load(f)

# Validate the embedding matrix
assert training_embeddings.shape == (TRAINING_SIZE, 512)
assert training_embeddings.dtype == np.float32
assert np.isfinite(training_embeddings).all()

# Validate the mapping
assert len(training_mapping) == TRAINING_SIZE
assert len({item["image_id"] for item in training_mapping}) == TRAINING_SIZE
assert [item["row"] for item in training_mapping] == list(range(TRAINING_SIZE))

# Validate correspondence with the original training records
assert [
    item["image_id"] for item in training_mapping
] == [
    record["id"] for record in sorted(training_records, key=lambda x: x["id"])
]

print("Training CLIP embeddings validation: PASSED")
print("Shape:", training_embeddings.shape)
print("Data type:", training_embeddings.dtype)
print("Mapping records:", len(training_mapping))
print("All values finite:", np.isfinite(training_embeddings).all())

Training CLIP embeddings validation: PASSED
Shape: (9900, 512)
Data type: float32
Mapping records: 9900
All values finite: True


In [21]:
VALIDATION_ROOT = PROCESSED_ROOT / "validation"

VALIDATION_IMAGES_DIR = VALIDATION_ROOT / "images"
VALIDATION_CAPTIONS_FILE = (
    VALIDATION_ROOT / "captions" / "captions_cleaned.json"
)

print("Validation images directory exists:",
      VALIDATION_IMAGES_DIR.exists())

print("Validation captions file exists:",
      VALIDATION_CAPTIONS_FILE.exists())

print("\nNumber of image files:",
      len(list(VALIDATION_IMAGES_DIR.glob("*.jpg")))
      if VALIDATION_IMAGES_DIR.exists() else "Directory not found")

Validation images directory exists: True
Validation captions file exists: True

Number of image files: 100


In [22]:
with open(VALIDATION_CAPTIONS_FILE, "r", encoding="utf-8") as f:
    validation_data = json.load(f)

print("Number of validation image records:",
      len(validation_data["images"]))

print("First validation image:")
print(validation_data["images"][0])

Number of validation image records: 100
First validation image:
{'license': 1, 'file_name': '000000377406.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000377406.jpg', 'height': 375, 'width': 500, 'date_captured': '2013-11-16 18:17:18', 'flickr_url': 'http://farm4.staticflickr.com/3061/3047321918_4aed197c01_z.jpg', 'id': 377406}


In [23]:
validation_records = validation_data["images"]

print("Validation records:", len(validation_records))
print("First record:", validation_records[0])

Validation records: 100
First record: {'license': 1, 'file_name': '000000377406.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000377406.jpg', 'height': 375, 'width': 500, 'date_captured': '2013-11-16 18:17:18', 'flickr_url': 'http://farm4.staticflickr.com/3061/3047321918_4aed197c01_z.jpg', 'id': 377406}


In [24]:
validation_ids = [record["id"] for record in validation_records]
validation_filenames = [
    record["file_name"] for record in validation_records
]

assert len(validation_records) == VALIDATION_SIZE
assert len(set(validation_ids)) == VALIDATION_SIZE
assert len(set(validation_filenames)) == VALIDATION_SIZE

missing_validation_images = [
    filename
    for filename in validation_filenames
    if not (VALIDATION_IMAGES_DIR / filename).exists()
]

assert len(missing_validation_images) == 0

# Ensure validation images do not overlap with training images
assert set(validation_ids).isdisjoint(set(training_ids))

print("Validation records validation: PASSED")
print("Unique image IDs:", len(set(validation_ids)))
print("Unique filenames:", len(set(validation_filenames)))
print("Missing images:", len(missing_validation_images))
print("Training/validation overlap:", 
      len(set(validation_ids) & set(training_ids)))

Validation records validation: PASSED
Unique image IDs: 100
Unique filenames: 100
Missing images: 0
Training/validation overlap: 0


In [25]:
validation_embedding_dir = (
    PROCESSED_ROOT / "validation" / "embeddings"
)

validation_summary = clip_module.generate_clip_image_embeddings(
    image_records=validation_records,
    image_dir=VALIDATION_IMAGES_DIR,
    output_dir=validation_embedding_dir,
    batch_size=16
)

validation_summary

Loading CLIP model: openai/clip-vit-base-patch32
Device: cpu


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating CLIP image embeddings:   0%|          | 0/7 [00:00<?, ?it/s]


CLIP embedding generation completed.
Shape: (100, 512)
Data type: float32
Embeddings saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\embeddings\clip_image_embeddings.npy
Mapping saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\embeddings\image_embedding_index.json


{'model_name': 'openai/clip-vit-base-patch32',
 'number_of_images': 100,
 'embedding_dimension': 512,
 'embedding_shape': [100, 512],
 'embedding_dtype': 'float32',
 'embedding_norm': 'L2',
 'normalized': True,
 'device': 'cpu',
 'batch_size': 16,
 'embeddings_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\validation\\embeddings\\clip_image_embeddings.npy',
 'mapping_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\validation\\embeddings\\image_embedding_index.json',
 'status': 'success'}

In [26]:
validation_embeddings = np.load(
    validation_embedding_dir / "clip_image_embeddings.npy"
)

with open(
    validation_embedding_dir / "image_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    validation_mapping = json.load(f)

assert validation_embeddings.shape == (VALIDATION_SIZE, 512)
assert validation_embeddings.dtype == np.float32
assert np.isfinite(validation_embeddings).all()

assert len(validation_mapping) == VALIDATION_SIZE
assert len({item["image_id"] for item in validation_mapping}) == VALIDATION_SIZE
assert [item["row"] for item in validation_mapping] == list(range(VALIDATION_SIZE))

assert [
    item["image_id"] for item in validation_mapping
] == [
    record["id"] for record in sorted(validation_records, key=lambda x: x["id"])
]

print("Validation CLIP embeddings validation: PASSED")
print("Shape:", validation_embeddings.shape)
print("Data type:", validation_embeddings.dtype)
print("Mapping records:", len(validation_mapping))
print("All values finite:", np.isfinite(validation_embeddings).all())

Validation CLIP embeddings validation: PASSED
Shape: (100, 512)
Data type: float32
Mapping records: 100
All values finite: True


In [27]:
TEST_ROOT = PROCESSED_ROOT / "test"

TEST_IMAGES_DIR = TEST_ROOT / "images"

TEST_CAPTIONS_FILE = (
    TEST_ROOT / "captions" / "captions_cleaned.json"
)

print("Test images directory exists:", TEST_IMAGES_DIR.exists())
print("Test captions file exists:", TEST_CAPTIONS_FILE.exists())

print(
    "Number of image files:",
    len(list(TEST_IMAGES_DIR.glob("*.jpg")))
)

Test images directory exists: True
Test captions file exists: True
Number of image files: 100


In [28]:
with open(TEST_CAPTIONS_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_records = test_data["images"]

print("Test records:", len(test_records))
print("First test record:", test_records[0])

Test records: 100
First test record: {'license': 3, 'file_name': '000000460347.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000460347.jpg', 'height': 640, 'width': 427, 'date_captured': '2013-11-17 00:11:09', 'flickr_url': 'http://farm9.staticflickr.com/8203/8262915638_911e6c6ea8_z.jpg', 'id': 460347}


In [29]:
test_ids = [record["id"] for record in test_records]

test_filenames = [
    record["file_name"] for record in test_records
]

assert len(test_records) == TEST_SIZE
assert len(set(test_ids)) == TEST_SIZE
assert len(set(test_filenames)) == TEST_SIZE

missing_test_images = [
    filename
    for filename in test_filenames
    if not (TEST_IMAGES_DIR / filename).exists()
]

assert len(missing_test_images) == 0

# Ensure no overlap with training or validation
assert set(test_ids).isdisjoint(set(training_ids))
assert set(test_ids).isdisjoint(set(validation_ids))

print("Test records validation: PASSED")
print("Unique image IDs:", len(set(test_ids)))
print("Unique filenames:", len(set(test_filenames)))
print("Missing images:", len(missing_test_images))
print("Training/test overlap:", len(set(test_ids) & set(training_ids)))
print("Validation/test overlap:", len(set(test_ids) & set(validation_ids)))

Test records validation: PASSED
Unique image IDs: 100
Unique filenames: 100
Missing images: 0
Training/test overlap: 0
Validation/test overlap: 0


In [30]:
test_embedding_dir = (
    PROCESSED_ROOT / "test" / "embeddings"
)

test_summary = clip_module.generate_clip_image_embeddings(
    image_records=test_records,
    image_dir=TEST_IMAGES_DIR,
    output_dir=test_embedding_dir,
    batch_size=16
)

test_summary

Loading CLIP model: openai/clip-vit-base-patch32
Device: cpu


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating CLIP image embeddings:   0%|          | 0/7 [00:00<?, ?it/s]


CLIP embedding generation completed.
Shape: (100, 512)
Data type: float32
Embeddings saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\embeddings\clip_image_embeddings.npy
Mapping saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\embeddings\image_embedding_index.json


{'model_name': 'openai/clip-vit-base-patch32',
 'number_of_images': 100,
 'embedding_dimension': 512,
 'embedding_shape': [100, 512],
 'embedding_dtype': 'float32',
 'embedding_norm': 'L2',
 'normalized': True,
 'device': 'cpu',
 'batch_size': 16,
 'embeddings_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\test\\embeddings\\clip_image_embeddings.npy',
 'mapping_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\test\\embeddings\\image_embedding_index.json',
 'status': 'success'}

In [31]:
test_embeddings = np.load(
    test_embedding_dir / "clip_image_embeddings.npy"
)

with open(
    test_embedding_dir / "image_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    test_mapping = json.load(f)

assert test_embeddings.shape == (TEST_SIZE, 512)
assert test_embeddings.dtype == np.float32
assert np.isfinite(test_embeddings).all()

assert len(test_mapping) == TEST_SIZE
assert len({item["image_id"] for item in test_mapping}) == TEST_SIZE
assert [item["row"] for item in test_mapping] == list(range(TEST_SIZE))

assert [
    item["image_id"] for item in test_mapping
] == [
    record["id"] for record in sorted(test_records, key=lambda x: x["id"])
]

print("Test CLIP embeddings validation: PASSED")
print("Shape:", test_embeddings.shape)
print("Data type:", test_embeddings.dtype)
print("Mapping records:", len(test_mapping))
print("All values finite:", np.isfinite(test_embeddings).all())

Test CLIP embeddings validation: PASSED
Shape: (100, 512)
Data type: float32
Mapping records: 100
All values finite: True


In [32]:
import numpy as np

split_dirs = {
    "training": PROCESSED_ROOT / "training" / "embeddings",
    "validation": PROCESSED_ROOT / "validation" / "embeddings",
    "test": PROCESSED_ROOT / "test" / "embeddings",
}

expected_counts = {
    "training": TRAINING_SIZE,
    "validation": VALIDATION_SIZE,
    "test": TEST_SIZE,
}

for split, directory in split_dirs.items():
    embeddings = np.load(
        directory / "clip_image_embeddings.npy",
        mmap_mode="r"
    )

    norms = np.linalg.norm(embeddings, ord=2, axis=1)

    assert embeddings.shape == (expected_counts[split], 512)
    assert embeddings.dtype == np.float32
    assert np.isfinite(embeddings).all()
    assert np.allclose(norms, 1.0, atol=1e-5)

    print(f"{split.upper()}: PASSED")
    print("  Shape:", embeddings.shape)
    print("  Minimum norm:", norms.min())
    print("  Maximum norm:", norms.max())

TRAINING: PASSED
  Shape: (9900, 512)
  Minimum norm: 0.9999997
  Maximum norm: 1.0000002
VALIDATION: PASSED
  Shape: (100, 512)
  Minimum norm: 0.99999976
  Maximum norm: 1.0000001
TEST: PASSED
  Shape: (100, 512)
  Minimum norm: 0.9999998
  Maximum norm: 1.0000001


In [33]:
import importlib
import embeddings.sbert_caption_embeddings as sbert_module

importlib.reload(sbert_module)

print("Sentence-BERT module loaded successfully.")

Sentence-BERT module loaded successfully.


In [34]:
training_caption_records = captions_data["annotations"]

print("Number of training captions:",
      len(training_caption_records))

print("First caption record:")
print(training_caption_records[0])

Number of training captions: 49500
First caption record:
{'image_id': 149, 'id': 481005, 'caption': 'People gathered outside in a big field on a cloudy day'}


In [35]:
# Number of captions
assert len(training_caption_records) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# Required fields
assert all(
    {"id", "image_id", "caption"}.issubset(record)
    for record in training_caption_records
)

# Unique caption IDs
assert len({
    record["id"] for record in training_caption_records
}) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# No empty captions
assert all(
    isinstance(record["caption"], str)
    and record["caption"].strip()
    for record in training_caption_records
)

print("Training caption records validation: PASSED")
print("Number of captions:", len(training_caption_records))
print(
    "Unique caption IDs:",
    len({record["id"] for record in training_caption_records})
)

Training caption records validation: PASSED
Number of captions: 49500
Unique caption IDs: 49500


In [76]:
sbert_sample_dir = (
    PROCESSED_ROOT
    / "embedding_experiments"
    / "sbert_sample"
)

sbert_sample_summary = (
    sbert_module.generate_sbert_caption_embeddings(
        caption_records=training_caption_records[:10],
        output_dir=sbert_sample_dir,
        batch_size=4
    )
)

sbert_sample_summary

Loading Sentence-BERT model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\01_src\embeddings\sbert_caption_embeddings.py:97: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Sentence-BERT embedding generation completed.
Shape: (10, 384)
Data type: float32
Embeddings saved: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\sbert_sample\sbert_caption_embeddings.npy
Mapping saved: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\sbert_sample\caption_embedding_index.json


{'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'number_of_captions': 10,
 'embedding_dimension': 384,
 'embedding_shape': [10, 384],
 'embedding_dtype': 'float32',
 'normalized': True,
 'normalization_method': 'L2',
 'device': 'cpu',
 'batch_size': 4,
 'minimum_l2_norm': 0.9999999403953552,
 'maximum_l2_norm': 1.0,
 'embeddings_file': 'C:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\sbert_sample\\sbert_caption_embeddings.npy',
 'mapping_file': 'C:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\embedding_experiments\\sbert_sample\\caption_embedding_index.json',
 'status': 'success'}

In [37]:
# Restore the Sentence-BERT sample output directory
sbert_sample_dir = (
    PROCESSED_ROOT
    / "embedding_experiments"
    / "sbert_sample"
)

assert sbert_sample_dir.exists(), (
    f"Sample directory not found: {sbert_sample_dir}"
)

print("Sentence-BERT sample directory:", sbert_sample_dir)
print("Directory exists:", sbert_sample_dir.exists())

Sentence-BERT sample directory: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\embedding_experiments\sbert_sample
Directory exists: True


In [38]:
sbert_sample_embeddings = np.load(
    sbert_sample_dir / "sbert_caption_embeddings.npy"
)

with open(
    sbert_sample_dir / "caption_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    sbert_sample_mapping = json.load(f)

sample_norms = np.linalg.norm(
    sbert_sample_embeddings,
    ord=2,
    axis=1
)

assert sbert_sample_embeddings.shape == (10, 384)
assert sbert_sample_embeddings.dtype == np.float32
assert np.isfinite(sbert_sample_embeddings).all()
assert np.allclose(sample_norms, 1.0, atol=1e-5)

assert len(sbert_sample_mapping) == 10
assert [item["row"] for item in sbert_sample_mapping] == list(range(10))

assert [
    item["caption_id"] for item in sbert_sample_mapping
] == [
    record["id"]
    for record in sorted(
        training_caption_records[:10],
        key=lambda x: x["id"]
    )
]

print("Sentence-BERT sample validation: PASSED")
print("Shape:", sbert_sample_embeddings.shape)
print("Data type:", sbert_sample_embeddings.dtype)
print("Minimum L2 norm:", sample_norms.min())
print("Maximum L2 norm:", sample_norms.max())
print("Mapping records:", len(sbert_sample_mapping))

Sentence-BERT sample validation: PASSED
Shape: (10, 384)
Data type: float32
Minimum L2 norm: 0.99999994
Maximum L2 norm: 1.0
Mapping records: 10


In [39]:
# Load saved embeddings
sbert_sample_embeddings = np.load(
    sbert_sample_dir / "sbert_caption_embeddings.npy"
)

# Load caption mapping
with open(
    sbert_sample_dir / "caption_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    sbert_sample_mapping = json.load(f)

# Calculate L2 norms
sample_norms = np.linalg.norm(
    sbert_sample_embeddings,
    ord=2,
    axis=1
)

# Validate embedding matrix
assert sbert_sample_embeddings.shape == (10, 384)
assert sbert_sample_embeddings.dtype == np.float32
assert np.isfinite(sbert_sample_embeddings).all()
assert np.allclose(sample_norms, 1.0, atol=1e-5)

# Validate mapping
assert len(sbert_sample_mapping) == 10

assert [
    item["row"] for item in sbert_sample_mapping
] == list(range(10))

# Check caption IDs
expected_records = sorted(
    training_caption_records[:10],
    key=lambda x: x["id"]
)

assert [
    item["caption_id"] for item in sbert_sample_mapping
] == [
    record["id"] for record in expected_records
]

# Check image IDs
assert [
    item["image_id"] for item in sbert_sample_mapping
] == [
    record["image_id"] for record in expected_records
]

print("Sentence-BERT sample validation: PASSED")
print("Shape:", sbert_sample_embeddings.shape)
print("Data type:", sbert_sample_embeddings.dtype)
print("Minimum L2 norm:", sample_norms.min())
print("Maximum L2 norm:", sample_norms.max())
print("Mapping records:", len(sbert_sample_mapping))

Sentence-BERT sample validation: PASSED
Shape: (10, 384)
Data type: float32
Minimum L2 norm: 0.99999994
Maximum L2 norm: 1.0
Mapping records: 10


In [40]:
# Final destination for training embeddings
training_embeddings_dir = (
    PROCESSED_ROOT
    / "training"
    / "embeddings"
)

training_embeddings_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Verify the input and output
print("Number of training captions:", len(training_caption_records))
print("Output directory:", training_embeddings_dir)

assert len(training_caption_records) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

print("Preparation: PASSED")

Number of training captions: 49500
Output directory: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings
Preparation: PASSED


In [41]:
sbert_training_summary = (
    sbert_module.generate_sbert_caption_embeddings(
        caption_records=training_caption_records,
        output_dir=training_embeddings_dir,
        batch_size=64,
        device="cpu"
    )
)

sbert_training_summary

Loading Sentence-BERT model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\01_src\embeddings\sbert_caption_embeddings.py:97: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/774 [00:00<?, ?it/s]


Sentence-BERT embedding generation completed.
Shape: (49500, 384)
Data type: float32
Embeddings saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings\sbert_caption_embeddings.npy
Mapping saved: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings\caption_embedding_index.json


{'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'number_of_captions': 49500,
 'embedding_dimension': 384,
 'embedding_shape': [49500, 384],
 'embedding_dtype': 'float32',
 'normalized': True,
 'normalization_method': 'L2',
 'device': 'cpu',
 'batch_size': 64,
 'minimum_l2_norm': 0.9999998807907104,
 'maximum_l2_norm': 1.0000001192092896,
 'embeddings_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\training\\embeddings\\sbert_caption_embeddings.npy',
 'mapping_file': 'c:\\Users\\iso99\\Desktop\\Ahmed\\current\\Image-Captioning-RAG\\02_processed_data\\training\\embeddings\\caption_embedding_index.json',
 'status': 'success'}

In [42]:
# Load the complete Sentence-BERT embedding matrix
sbert_training_embeddings = np.load(
    training_embeddings_dir / "sbert_caption_embeddings.npy"
)

# Load the caption-to-image mapping
with open(
    training_embeddings_dir / "caption_embedding_index.json",
    "r",
    encoding="utf-8"
) as f:
    sbert_training_mapping = json.load(f)

print("Embeddings loaded:", sbert_training_embeddings.shape)
print("Mapping records:", len(sbert_training_mapping))

Embeddings loaded: (49500, 384)
Mapping records: 49500


In [82]:
# Expected matrix dimensions
assert sbert_training_embeddings.shape == (TRAINING_SIZE * CAPTIONS_PER_IMAGE, 384)

# Expected data type
assert sbert_training_embeddings.dtype == np.float32

# Check for NaN or infinite values
assert np.isfinite(sbert_training_embeddings).all()

# Calculate L2 norms
sbert_training_norms = np.linalg.norm(
    sbert_training_embeddings,
    ord=2,
    axis=1
)

# Verify L2 normalization
assert np.allclose(
    sbert_training_norms,
    1.0,
    atol=1e-5
)

print("Embedding matrix validation: PASSED")
print("Shape:", sbert_training_embeddings.shape)
print("Data type:", sbert_training_embeddings.dtype)
print("Minimum L2 norm:", sbert_training_norms.min())
print("Maximum L2 norm:", sbert_training_norms.max())

Embedding matrix validation: PASSED
Shape: (49500, 384)
Data type: float32
Minimum L2 norm: 0.9999999
Maximum L2 norm: 1.0000001


In [83]:
# Expected number of mapping records
assert len(sbert_training_mapping) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# Verify sequential row numbers
assert [
    item["row"] for item in sbert_training_mapping
] == list(range(TRAINING_SIZE * CAPTIONS_PER_IMAGE))

# Sort original caption records by COCO caption ID
expected_training_records = sorted(
    training_caption_records,
    key=lambda x: x["id"]
)

# Verify caption IDs
assert [
    item["caption_id"] for item in sbert_training_mapping
] == [
    record["id"] for record in expected_training_records
]

# Verify associated image IDs
assert [
    item["image_id"] for item in sbert_training_mapping
] == [
    record["image_id"] for record in expected_training_records
]

# Verify caption ID uniqueness
assert len({
    item["caption_id"] for item in sbert_training_mapping
}) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

print("Caption mapping validation: PASSED")
print("Mapping records:", len(sbert_training_mapping))
print("Unique caption IDs:", len({
    item["caption_id"] for item in sbert_training_mapping
}))

Caption mapping validation: PASSED
Mapping records: 49500
Unique caption IDs: 49500


In [84]:
print("=" * 55)
print("SENTENCE-BERT TRAINING EMBEDDINGS")
print("=" * 55)

print("Embedding generation: PASSED")
print("Matrix validation: PASSED")
print("Mapping validation: PASSED")

print("\nFinal matrix shape:", sbert_training_embeddings.shape)
print("Total caption embeddings:", len(sbert_training_mapping))
print("Embedding dimension:", sbert_training_embeddings.shape[1])
print("Normalization: L2")
print("Data type:", sbert_training_embeddings.dtype)

SENTENCE-BERT TRAINING EMBEDDINGS
Embedding generation: PASSED
Matrix validation: PASSED
Mapping validation: PASSED

Final matrix shape: (49500, 384)
Total caption embeddings: 49500
Embedding dimension: 384
Normalization: L2
Data type: float32


In [85]:
import json
import platform
from importlib.metadata import PackageNotFoundError, version

# Retrieve installed package versions
def get_package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not installed"


embedding_metadata = {
    "pipeline_configuration": get_pipeline_snapshot(),
    "metadata_version": "1.0",
    "project": "Image Captioning RAG",
    "stage": "Phase 1 - Embedding Generation",
    "dataset": "MS COCO 2017",
    "split": "training",

    "dataset_statistics": {
        "number_of_images": TRAINING_SIZE,
        "number_of_captions": TRAINING_SIZE * CAPTIONS_PER_IMAGE
    },

    "image_embeddings": {
        "model_name": "openai/clip-vit-base-patch32",
        "encoder": "CLIP image encoder",
        "number_of_images": TRAINING_SIZE,
        "embedding_dimension": 512,
        "embedding_shape": [TRAINING_SIZE, 512],
        "data_type": "float32",
        "normalized": True,
        "normalization_method": "L2",
        "device": "cpu",
        "batch_size": 16,
        "input_preprocessing": {
            "image_mode": "RGB",
            "exif_orientation_correction": True,
            "processor": "Official CLIP image processor",
            "output_pixel_shape": [1, 3, 224, 224]
        },
        "files": {
            "embeddings": "clip_image_embeddings.npy",
            "mapping": "image_embedding_index.json"
        },
        "validation": {
            "status": "passed",
            "expected_shape": [TRAINING_SIZE, 512],
            "all_values_finite": True,
            "l2_normalization_verified": True,
            "mapping_verified": True
        }
    },

    "caption_embeddings": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "encoder": "Sentence-BERT",
        "number_of_captions": TRAINING_SIZE * CAPTIONS_PER_IMAGE,
        "embedding_dimension": 384,
        "embedding_shape": [TRAINING_SIZE * CAPTIONS_PER_IMAGE, 384],
        "data_type": "float32",
        "normalized": True,
        "normalization_method": "L2",
        "device": "cpu",
        "batch_size": 64,
        "text_preprocessing": {
            "source": "captions_cleaned.json",
            "method": "Conservative whitespace cleanup",
            "preserve_capitalization": True,
            "preserve_punctuation": True,
            "preserve_word_order": True,
            "stopword_removal": False,
            "stemming": False,
            "lemmatization": False
        },
        "files": {
            "embeddings": "sbert_caption_embeddings.npy",
            "mapping": "caption_embedding_index.json"
        },
        "validation": {
            "status": "passed",
            "expected_shape": [TRAINING_SIZE * CAPTIONS_PER_IMAGE, 384],
            "all_values_finite": True,
            "l2_normalization_verified": True,
            "mapping_verified": True,
            "unique_caption_ids_verified": True,
            "image_id_associations_verified": True
        }
    },

    "mapping_information": {
        "image_mapping_fields": [
            "row",
            "image_id"
        ],
        "caption_mapping_fields": [
            "row",
            "caption_id",
            "image_id"
        ],
        "row_indexing": "Zero-based",
        "purpose": "Traceability between embedding rows and original dataset records"
    },

    "software_environment": {
        "python": platform.python_version(),
        "pytorch": get_package_version("torch"),
        "transformers": get_package_version("transformers"),
        "sentence_transformers": get_package_version("sentence-transformers"),
        "numpy": get_package_version("numpy"),
        "pillow": get_package_version("Pillow")
    },

    "overall_validation_status": "passed"
}

# Save metadata
metadata_file = training_embeddings_dir / "embedding_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(
        embedding_metadata,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Embedding metadata saved successfully.")
print("File:", metadata_file)
print("Overall validation:", embedding_metadata["overall_validation_status"])

Embedding metadata saved successfully.
File: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\embeddings\embedding_metadata.json
Overall validation: passed


In [86]:
with open(metadata_file, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

assert saved_metadata["split"] == "training"

assert saved_metadata["image_embeddings"]["embedding_shape"] == [
    TRAINING_SIZE, 512
]

assert saved_metadata["caption_embeddings"]["embedding_shape"] == [
    TRAINING_SIZE * CAPTIONS_PER_IMAGE, 384
]

assert saved_metadata["image_embeddings"]["validation"]["status"] == "passed"
assert saved_metadata["caption_embeddings"]["validation"]["status"] == "passed"

assert saved_metadata["overall_validation_status"] == "passed"

print("Metadata validation: PASSED")
print("Metadata version:", saved_metadata["metadata_version"])
print("CLIP images:", saved_metadata["image_embeddings"]["number_of_images"])
print("SBERT captions:", saved_metadata["caption_embeddings"]["number_of_captions"])
print("Python:", saved_metadata["software_environment"]["python"])
print("PyTorch:", saved_metadata["software_environment"]["pytorch"])
check_pipeline_compatibility(
    saved_metadata["pipeline_configuration"],
    "Embedding metadata",
)


Metadata validation: PASSED
Metadata version: 1.0
CLIP images: 9900
SBERT captions: 49500
Python: 3.11.4
PyTorch: 2.15.0


In [87]:
validation_metadata = {
    "metadata_version": "1.0",
    "project": "Image Captioning RAG",
    "stage": "Phase 1 - Embedding Generation",
    "dataset": "MS COCO 2017",
    "split": "validation",

    "dataset_statistics": {
        "number_of_images": 100
    },

    "image_embeddings": {
        "model_name": "openai/clip-vit-base-patch32",
        "encoder": "CLIP image encoder",
        "number_of_images": 100,
        "embedding_dimension": 512,
        "embedding_shape": [100, 512],
        "data_type": "float32",
        "normalized": True,
        "normalization_method": "L2",
        "device": "cpu",
        "batch_size": 16,

        "input_preprocessing": {
            "image_mode": "RGB",
            "exif_orientation_correction": True,
            "processor": "Official CLIP image processor",
            "output_pixel_shape": [1, 3, 224, 224]
        },

        "files": {
            "embeddings": "clip_image_embeddings.npy",
            "mapping": "image_embedding_index.json"
        },

        "validation": {
            "status": "passed",
            "expected_shape": [100, 512],
            "all_values_finite": True,
            "l2_normalization_verified": True,
            "minimum_l2_norm": 0.99999976,
            "maximum_l2_norm": 1.0000001
        }
    },

    "software_environment": {
        "python": platform.python_version(),
        "pytorch": get_package_version("torch"),
        "transformers": get_package_version("transformers"),
        "numpy": get_package_version("numpy"),
        "pillow": get_package_version("Pillow")
    },

    "overall_validation_status": "passed"
}

validation_embeddings_dir = (
    PROCESSED_ROOT / "validation" / "embeddings"
)

validation_metadata_file = (
    validation_embeddings_dir / "embedding_metadata.json"
)

with open(validation_metadata_file, "w", encoding="utf-8") as f:
    json.dump(
        validation_metadata,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Validation metadata saved successfully.")
print("File:", validation_metadata_file)

Validation metadata saved successfully.
File: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\embeddings\embedding_metadata.json


In [88]:
test_metadata = {
    "metadata_version": "1.0",
    "project": "Image Captioning RAG",
    "stage": "Phase 1 - Embedding Generation",
    "dataset": "MS COCO 2017",
    "split": "test",

    "dataset_statistics": {
        "number_of_images": 100
    },

    "image_embeddings": {
        "model_name": "openai/clip-vit-base-patch32",
        "encoder": "CLIP image encoder",
        "number_of_images": 100,
        "embedding_dimension": 512,
        "embedding_shape": [100, 512],
        "data_type": "float32",
        "normalized": True,
        "normalization_method": "L2",
        "device": "cpu",
        "batch_size": 16,

        "input_preprocessing": {
            "image_mode": "RGB",
            "exif_orientation_correction": True,
            "processor": "Official CLIP image processor",
            "output_pixel_shape": [1, 3, 224, 224]
        },

        "files": {
            "embeddings": "clip_image_embeddings.npy",
            "mapping": "image_embedding_index.json"
        },

        "validation": {
            "status": "passed",
            "expected_shape": [100, 512],
            "all_values_finite": True,
            "l2_normalization_verified": True,
            "minimum_l2_norm": 0.9999998,
            "maximum_l2_norm": 1.0000001
        }
    },

    "software_environment": {
        "python": platform.python_version(),
        "pytorch": get_package_version("torch"),
        "transformers": get_package_version("transformers"),
        "numpy": get_package_version("numpy"),
        "pillow": get_package_version("Pillow")
    },

    "overall_validation_status": "passed"
}

test_embeddings_dir = (
    PROCESSED_ROOT / "test" / "embeddings"
)

test_metadata_file = (
    test_embeddings_dir / "embedding_metadata.json"
)

with open(test_metadata_file, "w", encoding="utf-8") as f:
    json.dump(
        test_metadata,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Test metadata saved successfully.")
print("File:", test_metadata_file)

Test metadata saved successfully.
File: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\embeddings\embedding_metadata.json


In [89]:
# Reload both metadata files
with open(validation_metadata_file, "r", encoding="utf-8") as f:
    saved_validation_metadata = json.load(f)

with open(test_metadata_file, "r", encoding="utf-8") as f:
    saved_test_metadata = json.load(f)

# Validate split identification
assert saved_validation_metadata["split"] == "validation"
assert saved_test_metadata["split"] == "test"

# Validate model information
assert (
    saved_validation_metadata["image_embeddings"]["model_name"]
    == "openai/clip-vit-base-patch32"
)

assert (
    saved_test_metadata["image_embeddings"]["model_name"]
    == "openai/clip-vit-base-patch32"
)

# Validate embedding dimensions
assert saved_validation_metadata["image_embeddings"]["embedding_shape"] == [100, 512]
assert saved_test_metadata["image_embeddings"]["embedding_shape"] == [100, 512]

# Validate normalization
assert saved_validation_metadata["image_embeddings"]["normalized"] is True
assert saved_test_metadata["image_embeddings"]["normalized"] is True

# Validate overall status
assert saved_validation_metadata["overall_validation_status"] == "passed"
assert saved_test_metadata["overall_validation_status"] == "passed"

print("Validation metadata check: PASSED")
print("Test metadata check: PASSED")

Validation metadata check: PASSED
Test metadata check: PASSED


In [90]:
from pathlib import Path

# Knowledge-base root
KNOWLEDGE_BASE_ROOT = PROCESSED_ROOT / "knowledge_base"

# Required directories
knowledge_base_directories = [
    KNOWLEDGE_BASE_ROOT / "indexes" / "clip_image_index",
    KNOWLEDGE_BASE_ROOT / "indexes" / "sbert_caption_index",
    KNOWLEDGE_BASE_ROOT / "documents",
    KNOWLEDGE_BASE_ROOT / "metadata",
    KNOWLEDGE_BASE_ROOT / "manifest",
]

# Create directories
for directory in knowledge_base_directories:
    directory.mkdir(parents=True, exist_ok=True)

# Verify directory creation
for directory in knowledge_base_directories:
    assert directory.exists() and directory.is_dir()
    print("Created:", directory.relative_to(PROCESSED_ROOT))

print("\nKnowledge-base directory structure: PASSED")

Created: knowledge_base\indexes\clip_image_index
Created: knowledge_base\indexes\sbert_caption_index
Created: knowledge_base\documents
Created: knowledge_base\metadata
Created: knowledge_base\manifest

Knowledge-base directory structure: PASSED


In [91]:
%pip install faiss-cpu


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.3 MB 3.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.3 MB 3.9 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/16.3 MB 2.3 MB/s eta 0:00:07
   ------- -------------------------------- 3.1/16.3 MB 3.2 MB/s eta 0:00:05
   ---------- ----------------------------- 4.5/16.3 MB 3.8 MB/s eta 0:00:04
   -------------- ------------------------- 5.8/16.3 MB 4.2 MB/s eta 0:00:03
   ---------------- ----------------------- 6.8/16.3 MB 4.4 MB/s eta 0:00:03
   ------------------- -------------------- 7.9/16.3 MB 4.3 MB/s eta 0:00:02
   --------------------- ------------------ 8.9/16.3 MB 4.4 MB/s eta 0:00:02
   ------------------------- -------------- 10.5/16.3 MB 4.7 MB/s eta 0:00:02
   ---------------------------- ----------- 11.8/16.3 MB 4.8 MB/s eta 0:00:01
   -------

In [92]:
import faiss

print("FAISS version:", faiss.__version__)
print("FAISS installation: PASSED")

FAISS version: 1.15.1
FAISS installation: PASSED


In [93]:
import faiss
print("FAISS imported successfully.")

FAISS imported successfully.


In [94]:
import json
import numpy as np
from pathlib import Path

# Paths to the training CLIP embeddings
TRAINING_EMBEDDINGS_DIR = (
    PROCESSED_ROOT / "training" / "embeddings"
)

CLIP_EMBEDDINGS_PATH = (
    TRAINING_EMBEDDINGS_DIR / "clip_image_embeddings.npy"
)

CLIP_MAPPING_PATH = (
    TRAINING_EMBEDDINGS_DIR / "image_embedding_index.json"
)

# Check that both files exist
assert CLIP_EMBEDDINGS_PATH.exists(), (
    f"Missing file: {CLIP_EMBEDDINGS_PATH}"
)

assert CLIP_MAPPING_PATH.exists(), (
    f"Missing file: {CLIP_MAPPING_PATH}"
)

# Load embeddings and mapping
clip_embeddings = np.load(CLIP_EMBEDDINGS_PATH)

with open(CLIP_MAPPING_PATH, "r", encoding="utf-8") as f:
    clip_mapping = json.load(f)

# Basic validation
assert clip_embeddings.shape == (TRAINING_SIZE, 512)
assert clip_embeddings.dtype == np.float32
assert np.isfinite(clip_embeddings).all()
assert len(clip_mapping) == TRAINING_SIZE

# Display information
print("CLIP embeddings loaded successfully.")
print("Embedding shape:", clip_embeddings.shape)
print("Embedding dtype:", clip_embeddings.dtype)
print("Number of mapping records:", len(clip_mapping))

print("\nFirst mapping record:")
print(clip_mapping[0])

print("\nCLIP embedding input validation: PASSED")

CLIP embeddings loaded successfully.
Embedding shape: (9900, 512)
Embedding dtype: float32
Number of mapping records: 9900

First mapping record:
{'row': 0, 'image_id': 149, 'file_name': '000000000149.jpg'}

CLIP embedding input validation: PASSED


In [95]:
import faiss
import numpy as np
import time

# Output directory
CLIP_INDEX_DIR = (
    KNOWLEDGE_BASE_ROOT
    / "indexes"
    / "clip_image_index"
)

CLIP_INDEX_DIR.mkdir(parents=True, exist_ok=True)

CLIP_INDEX_PATH = (
    CLIP_INDEX_DIR / "clip_image_index.faiss"
)

# Verify mapping row order
assert all(
    record["row"] == i
    for i, record in enumerate(clip_mapping)
), "Mapping rows are not sequential."

# Prepare vectors for FAISS
clip_vectors = np.ascontiguousarray(
    clip_embeddings,
    dtype=np.float32
)

# Create an exact inner-product index
dimension = clip_vectors.shape[1]
clip_index = faiss.IndexFlatIP(dimension)

# Add all training image embeddings
start_time = time.perf_counter()

clip_index.add(clip_vectors)

indexing_time = time.perf_counter() - start_time

# Save the index
faiss.write_index(
    clip_index,
    str(CLIP_INDEX_PATH)
)

# Validate the constructed index
assert clip_index.ntotal == TRAINING_SIZE
assert clip_index.d == 512
assert clip_index.metric_type == faiss.METRIC_INNER_PRODUCT
assert CLIP_INDEX_PATH.exists()

print("CLIP FAISS index created successfully.")
print("Index type:", type(clip_index).__name__)
print("Number of indexed vectors:", clip_index.ntotal)
print("Vector dimension:", clip_index.d)
print("Similarity metric: Inner Product")
print(f"Index construction time: {indexing_time:.4f} seconds")
print("Saved to:", CLIP_INDEX_PATH)
print("\nCLIP index construction: PASSED")

CLIP FAISS index created successfully.
Index type: IndexFlatIP
Number of indexed vectors: 9900
Vector dimension: 512
Similarity metric: Inner Product
Index construction time: 0.0126 seconds
Saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\knowledge_base\indexes\clip_image_index\clip_image_index.faiss

CLIP index construction: PASSED


In [96]:
import faiss

# Reload the index from disk
loaded_clip_index = faiss.read_index(
    str(CLIP_INDEX_PATH)
)

# Validate the reloaded index
assert loaded_clip_index.ntotal == TRAINING_SIZE
assert loaded_clip_index.d == 512
assert loaded_clip_index.metric_type == faiss.METRIC_INNER_PRODUCT

# Display results
print("CLIP index reloaded successfully.")
print("Index type:", type(loaded_clip_index).__name__)
print("Number of vectors:", loaded_clip_index.ntotal)
print("Vector dimension:", loaded_clip_index.d)
print("Metric type:", loaded_clip_index.metric_type)
print("Index file size:", round(CLIP_INDEX_PATH.stat().st_size / (1024 ** 2), 2), "MB")

print("\nSaved CLIP index validation: PASSED")

CLIP index reloaded successfully.
Index type: IndexFlatIP
Number of vectors: 9900
Vector dimension: 512
Metric type: 0
Index file size: 19.34 MB

Saved CLIP index validation: PASSED


In [97]:
import json
import numpy as np

# Paths to training SBERT embeddings
SBERT_EMBEDDINGS_PATH = (
    TRAINING_EMBEDDINGS_DIR / "sbert_caption_embeddings.npy"
)

SBERT_MAPPING_PATH = (
    TRAINING_EMBEDDINGS_DIR / "caption_embedding_index.json"
)

# Check that both files exist
assert SBERT_EMBEDDINGS_PATH.exists(), (
    f"Missing file: {SBERT_EMBEDDINGS_PATH}"
)

assert SBERT_MAPPING_PATH.exists(), (
    f"Missing file: {SBERT_MAPPING_PATH}"
)

# Load embeddings and mapping
sbert_embeddings = np.load(SBERT_EMBEDDINGS_PATH)

with open(SBERT_MAPPING_PATH, "r", encoding="utf-8") as f:
    sbert_mapping = json.load(f)

# Validate dimensions and data
assert sbert_embeddings.shape == (TRAINING_SIZE * CAPTIONS_PER_IMAGE, 384)
assert sbert_embeddings.dtype == np.float32
assert np.isfinite(sbert_embeddings).all()
assert len(sbert_mapping) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# Validate row order
assert all(
    record["row"] == i
    for i, record in enumerate(sbert_mapping)
), "Mapping rows are not sequential."

# Display information
print("SBERT embeddings loaded successfully.")
print("Embedding shape:", sbert_embeddings.shape)
print("Embedding dtype:", sbert_embeddings.dtype)
print("Number of mapping records:", len(sbert_mapping))

print("\nFirst mapping record:")
print(sbert_mapping[0])

print("\nSBERT embedding input validation: PASSED")

SBERT embeddings loaded successfully.
Embedding shape: (49500, 384)
Embedding dtype: float32
Number of mapping records: 49500

First mapping record:
{'row': 0, 'caption_id': 51, 'image_id': 12902}

SBERT embedding input validation: PASSED


In [98]:
import faiss
import numpy as np
import time

# Output directory
SBERT_INDEX_DIR = (
    KNOWLEDGE_BASE_ROOT
    / "indexes"
    / "sbert_caption_index"
)

SBERT_INDEX_DIR.mkdir(parents=True, exist_ok=True)

SBERT_INDEX_PATH = (
    SBERT_INDEX_DIR / "sbert_caption_index.faiss"
)

# Prepare vectors for FAISS
sbert_vectors = np.ascontiguousarray(
    sbert_embeddings,
    dtype=np.float32
)

# Create an exact inner-product index
dimension = sbert_vectors.shape[1]
sbert_index = faiss.IndexFlatIP(dimension)

# Add all training caption embeddings
start_time = time.perf_counter()

sbert_index.add(sbert_vectors)

indexing_time = time.perf_counter() - start_time

# Save the index
faiss.write_index(
    sbert_index,
    str(SBERT_INDEX_PATH)
)

# Validate the constructed index
assert sbert_index.ntotal == TRAINING_SIZE * CAPTIONS_PER_IMAGE
assert sbert_index.d == 384
assert sbert_index.metric_type == faiss.METRIC_INNER_PRODUCT
assert SBERT_INDEX_PATH.exists()

print("SBERT FAISS index created successfully.")
print("Index type:", type(sbert_index).__name__)
print("Number of indexed vectors:", sbert_index.ntotal)
print("Vector dimension:", sbert_index.d)
print("Similarity metric: Inner Product")
print(f"Index construction time: {indexing_time:.4f} seconds")
print("Saved to:", SBERT_INDEX_PATH)

print("\nSBERT index construction: PASSED")

SBERT FAISS index created successfully.
Index type: IndexFlatIP
Number of indexed vectors: 49500
Vector dimension: 384
Similarity metric: Inner Product
Index construction time: 0.0578 seconds
Saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\knowledge_base\indexes\sbert_caption_index\sbert_caption_index.faiss

SBERT index construction: PASSED


In [99]:
import faiss

# Reload the SBERT index from disk
loaded_sbert_index = faiss.read_index(
    str(SBERT_INDEX_PATH)
)

# Validate the reloaded index
assert loaded_sbert_index.ntotal == TRAINING_SIZE * CAPTIONS_PER_IMAGE
assert loaded_sbert_index.d == 384
assert loaded_sbert_index.metric_type == faiss.METRIC_INNER_PRODUCT

# Display results
print("SBERT index reloaded successfully.")
print("Index type:", type(loaded_sbert_index).__name__)
print("Number of vectors:", loaded_sbert_index.ntotal)
print("Vector dimension:", loaded_sbert_index.d)
print("Metric type:", loaded_sbert_index.metric_type)

print(
    "Index file size:",
    round(SBERT_INDEX_PATH.stat().st_size / (1024 ** 2), 2),
    "MB"
)

print("\nSaved SBERT index validation: PASSED")

SBERT index reloaded successfully.
Index type: IndexFlatIP
Number of vectors: 49500
Vector dimension: 384
Metric type: 0
Index file size: 72.51 MB

Saved SBERT index validation: PASSED


In [100]:
import json
from pathlib import Path

TRAINING_ROOT = PROCESSED_ROOT / "training"

# Locate the relevant source files
source_directories = [
    TRAINING_ROOT / "images",
    TRAINING_ROOT / "captions",
    TRAINING_ROOT / "annotations",
    TRAINING_ROOT / "metadata",
    TRAINING_ROOT / "embeddings",
]

for directory in source_directories:
    print(f"\n{directory.relative_to(PROCESSED_ROOT)}")

    if directory.exists():
        for file_path in sorted(directory.iterdir()):
            if file_path.is_file():
                print("  -", file_path.name)
    else:
        print("  Directory does not exist.")


training\images
  - 000000000149.jpg
  - 000000000257.jpg
  - 000000000322.jpg
  - 000000000397.jpg
  - 000000000486.jpg
  - 000000000562.jpg
  - 000000000625.jpg
  - 000000000629.jpg
  - 000000000650.jpg
  - 000000000690.jpg
  - 000000000693.jpg
  - 000000000699.jpg
  - 000000000706.jpg
  - 000000000714.jpg
  - 000000000730.jpg
  - 000000000772.jpg
  - 000000000810.jpg
  - 000000000813.jpg
  - 000000000925.jpg
  - 000000000934.jpg
  - 000000000960.jpg
  - 000000001059.jpg
  - 000000001108.jpg
  - 000000001139.jpg
  - 000000001153.jpg
  - 000000001171.jpg
  - 000000001237.jpg
  - 000000001390.jpg
  - 000000001404.jpg
  - 000000001408.jpg
  - 000000001501.jpg
  - 000000001590.jpg
  - 000000001647.jpg
  - 000000001697.jpg
  - 000000001709.jpg
  - 000000001757.jpg
  - 000000001852.jpg
  - 000000001864.jpg
  - 000000001895.jpg
  - 000000001911.jpg
  - 000000001942.jpg
  - 000000001997.jpg
  - 000000001999.jpg
  - 000000002014.jpg
  - 000000002068.jpg
  - 000000002114.jpg
  - 000000002184.

In [101]:
import json

files_to_inspect = [
    TRAINING_ROOT / "captions" / "captions_cleaned.json",
    TRAINING_ROOT / "metadata" / "image_metadata.json",
    TRAINING_ROOT / "metadata" / "caption_metadata.json",
]

for file_path in files_to_inspect:
    print("\n" + "=" * 70)
    print("FILE:", file_path.name)

    if not file_path.exists():
        print("File not found.")
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Data type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

        for key, value in data.items():
            print(f"\nExample for '{key}':")

            if isinstance(value, list) and value:
                print(json.dumps(value[0], indent=2, ensure_ascii=False))
            elif isinstance(value, dict):
                print(json.dumps(value, indent=2, ensure_ascii=False)[:1500])
            else:
                print(value)

    elif isinstance(data, list):
        print("Number of records:", len(data))

        if data:
            print("First record:")
            print(json.dumps(data[0], indent=2, ensure_ascii=False))


FILE: captions_cleaned.json
Data type: dict
Top-level keys: ['info', 'licenses', 'images', 'annotations']

Example for 'info':
{
  "description": "COCO 2017 Dataset",
  "url": "http://cocodataset.org",
  "version": "1.0",
  "year": 2017,
  "contributor": "COCO Consortium",
  "date_created": "2017/09/01"
}

Example for 'licenses':
{
  "url": "http://creativecommons.org/licenses/by-nc-sa/2.0/",
  "id": 1,
  "name": "Attribution-NonCommercial-ShareAlike License"
}

Example for 'images':
{
  "license": 3,
  "file_name": "000000184613.jpg",
  "coco_url": "http://images.cocodataset.org/train2017/000000184613.jpg",
  "height": 336,
  "width": 500,
  "date_captured": "2013-11-14 12:36:29",
  "flickr_url": "http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg",
  "id": 184613
}

Example for 'annotations':
{
  "image_id": 149,
  "id": 481005,
  "caption": "People gathered outside in a big field on a cloudy day"
}

FILE: image_metadata.json
Data type: dict
Top-level keys: ['split', 'im

In [102]:
import json

CAPTIONS_PATH = (
    TRAINING_ROOT / "captions" / "captions_cleaned.json"
)

with open(CAPTIONS_PATH, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

print("Number of images:", len(captions_data["images"]))
print("Number of caption annotations:", len(captions_data["annotations"]))

print("\nFirst image record:")
print(json.dumps(
    captions_data["images"][0],
    indent=2,
    ensure_ascii=False
))

print("\nFirst caption annotation:")
print(json.dumps(
    captions_data["annotations"][0],
    indent=2,
    ensure_ascii=False
))

Number of images: 9900
Number of caption annotations: 49500

First image record:
{
  "license": 3,
  "file_name": "000000184613.jpg",
  "coco_url": "http://images.cocodataset.org/train2017/000000184613.jpg",
  "height": 336,
  "width": 500,
  "date_captured": "2013-11-14 12:36:29",
  "flickr_url": "http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg",
  "id": 184613
}

First caption annotation:
{
  "image_id": 149,
  "id": 481005,
  "caption": "People gathered outside in a big field on a cloudy day"
}


In [103]:
import json

metadata_files = [
    TRAINING_ROOT / "metadata" / "image_metadata.json",
    TRAINING_ROOT / "metadata" / "caption_metadata.json",
]

for path in metadata_files:
    print("\n" + "=" * 60)
    print("FILE:", path)

    if not path.exists():
        print("NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Data type:", type(data).__name__)

    if isinstance(data, list):
        print("Number of records:", len(data))
        if data:
            print("First record:")
            print(json.dumps(data[0], indent=2, ensure_ascii=False))

    elif isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

        for key, value in data.items():
            if isinstance(value, list):
                print(f"\n{key}: {len(value)} records")
                if value:
                    print("First record:")
                    print(json.dumps(
                        value[0],
                        indent=2,
                        ensure_ascii=False
                    ))
                break


FILE: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata\image_metadata.json
Data type: dict
Top-level keys: ['split', 'image_count', 'images']

images: 9900 records
First record:
{
  "image_id": 184613,
  "file_name": "000000184613.jpg",
  "width": 500,
  "height": 336,
  "mode": "RGB",
  "format": "JPEG",
  "file_size_bytes": 182585,
  "exif_orientation": 1
}

FILE: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata\caption_metadata.json
Data type: dict
Top-level keys: ['split', 'caption_count', 'unique_caption_count', 'empty_caption_count', 'word_statistics', 'cleaning_operations']

cleaning_operations: 2 records
First record:
"Normalize consecutive whitespace"


In [104]:
import json

CAPTIONS_PATH = (
    TRAINING_ROOT / "captions" / "captions_cleaned.json"
)

with open(CAPTIONS_PATH, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

print("Number of images:", len(captions_data["images"]))
print("Number of annotations:", len(captions_data["annotations"]))

print("\nFirst image record:")
print(json.dumps(
    captions_data["images"][0],
    indent=2,
    ensure_ascii=False
))

print("\nFirst caption annotation:")
print(json.dumps(
    captions_data["annotations"][0],
    indent=2,
    ensure_ascii=False
))

print("\nSecond caption annotation:")
print(json.dumps(
    captions_data["annotations"][1],
    indent=2,
    ensure_ascii=False
))

Number of images: 9900
Number of annotations: 49500

First image record:
{
  "license": 3,
  "file_name": "000000184613.jpg",
  "coco_url": "http://images.cocodataset.org/train2017/000000184613.jpg",
  "height": 336,
  "width": 500,
  "date_captured": "2013-11-14 12:36:29",
  "flickr_url": "http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg",
  "id": 184613
}

First caption annotation:
{
  "image_id": 149,
  "id": 481005,
  "caption": "People gathered outside in a big field on a cloudy day"
}

Second caption annotation:
{
  "image_id": 149,
  "id": 481854,
  "caption": "People are flying kites on a high school field."
}


In [105]:
import json
from collections import defaultdict
from pathlib import Path

# --------------------------------------------------
# 1. Load source data
# --------------------------------------------------

with open(
    TRAINING_ROOT / "captions" / "captions_cleaned.json",
    "r",
    encoding="utf-8"
) as f:
    captions_data = json.load(f)

with open(
    TRAINING_ROOT / "metadata" / "image_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    image_metadata_data = json.load(f)

# Load embedding mappings if not already loaded
with open(CLIP_MAPPING_PATH, "r", encoding="utf-8") as f:
    clip_mapping = json.load(f)

with open(SBERT_MAPPING_PATH, "r", encoding="utf-8") as f:
    sbert_mapping = json.load(f)


# --------------------------------------------------
# 2. Prepare lookup dictionaries
# --------------------------------------------------

coco_images = captions_data["images"]
coco_captions = captions_data["annotations"]
image_metadata = image_metadata_data["images"]

# Image ID -> CLIP embedding information
clip_by_image_id = {
    record["image_id"]: record
    for record in clip_mapping
}

# Caption ID -> SBERT embedding information
sbert_by_caption_id = {
    record["caption_id"]: record
    for record in sbert_mapping
}

# Image ID -> image metadata
metadata_by_image_id = {
    record["image_id"]: record
    for record in image_metadata
}

# Image ID -> COCO image information
coco_image_by_id = {
    record["id"]: record
    for record in coco_images
}

# Image ID -> list of caption IDs
caption_ids_by_image = defaultdict(list)

for caption in coco_captions:
    caption_ids_by_image[caption["image_id"]].append(
        caption["id"]
    )


# --------------------------------------------------
# 3. Validate source relationships
# --------------------------------------------------

assert len(coco_images) == TRAINING_SIZE
assert len(coco_captions) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

assert len(clip_by_image_id) == TRAINING_SIZE
assert len(sbert_by_caption_id) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

assert len(metadata_by_image_id) == TRAINING_SIZE
assert len(coco_image_by_id) == TRAINING_SIZE

assert set(coco_image_by_id) == set(clip_by_image_id)
assert set(coco_image_by_id) == set(metadata_by_image_id)

assert set(
    caption["id"] for caption in coco_captions
) == set(sbert_by_caption_id)


# --------------------------------------------------
# 4. Build image documents
# --------------------------------------------------

image_documents = []

for image in coco_images:

    image_id = image["id"]

    clip_record = clip_by_image_id[image_id]
    metadata_record = metadata_by_image_id[image_id]

    # Confirm the image filename matches across sources
    assert image["file_name"] == clip_record["file_name"]
    assert image["file_name"] == metadata_record["file_name"]

    image_document = {
        "image_id": image_id,

        "file_name": image["file_name"],

        "embedding": {
            "model": "openai/clip-vit-base-patch32",
            "index": "clip_image_index",
            "row": clip_record["row"]
        },

        "caption_ids": sorted(
            caption_ids_by_image[image_id]
        ),

        "coco_metadata": image,

        "image_metadata": metadata_record
    }

    image_documents.append(image_document)


# --------------------------------------------------
# 5. Build caption documents
# --------------------------------------------------

caption_documents = []

for caption in coco_captions:

    caption_id = caption["id"]
    image_id = caption["image_id"]

    sbert_record = sbert_by_caption_id[caption_id]

    # Confirm caption-to-image association
    assert sbert_record["image_id"] == image_id

    caption_document = {
        "caption_id": caption_id,

        "image_id": image_id,

        "file_name": coco_image_by_id[image_id]["file_name"],

        "caption": caption["caption"],

        "embedding": {
            "model": "sentence-transformers/all-MiniLM-L6-v2",
            "index": "sbert_caption_index",
            "row": sbert_record["row"]
        }
    }

    caption_documents.append(caption_document)


# --------------------------------------------------
# 6. Validate document collections
# --------------------------------------------------

assert len(image_documents) == TRAINING_SIZE
assert len(caption_documents) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

assert len({
    doc["image_id"] for doc in image_documents
}) == TRAINING_SIZE

assert len({
    doc["caption_id"] for doc in caption_documents
}) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

assert all(
    len(doc["caption_ids"]) == 5
    for doc in image_documents
)

assert all(
    doc["image_id"] in coco_image_by_id
    for doc in caption_documents
)


# --------------------------------------------------
# 7. Save document collections
# --------------------------------------------------

DOCUMENTS_DIR = KNOWLEDGE_BASE_ROOT / "documents"
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DOCUMENTS_PATH = (
    DOCUMENTS_DIR / "image_documents.json"
)

CAPTION_DOCUMENTS_PATH = (
    DOCUMENTS_DIR / "caption_documents.json"
)

with open(
    IMAGE_DOCUMENTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        image_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    CAPTION_DOCUMENTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        caption_documents,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# 8. Display results
# --------------------------------------------------

print("Knowledge-base documents created successfully.")

print("\nImage documents:", len(image_documents))
print("Caption documents:", len(caption_documents))

print("\nFirst image document:")
print(json.dumps(
    image_documents[0],
    indent=2,
    ensure_ascii=False
))

print("\nFirst caption document:")
print(json.dumps(
    caption_documents[0],
    indent=2,
    ensure_ascii=False
))

print("\nSaved files:")
print(IMAGE_DOCUMENTS_PATH)
print(CAPTION_DOCUMENTS_PATH)

print("\nDocument construction: PASSED")

Knowledge-base documents created successfully.

Image documents: 9900
Caption documents: 49500

First image document:
{
  "image_id": 184613,
  "file_name": "000000184613.jpg",
  "embedding": {
    "model": "openai/clip-vit-base-patch32",
    "index": "clip_image_index",
    "row": 3163
  },
  "caption_ids": [
    474921,
    479322,
    479334,
    481560,
    483594
  ],
  "coco_metadata": {
    "license": 3,
    "file_name": "000000184613.jpg",
    "coco_url": "http://images.cocodataset.org/train2017/000000184613.jpg",
    "height": 336,
    "width": 500,
    "date_captured": "2013-11-14 12:36:29",
    "flickr_url": "http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg",
    "id": 184613
  },
  "image_metadata": {
    "image_id": 184613,
    "file_name": "000000184613.jpg",
    "width": 500,
    "height": 336,
    "mode": "RGB",
    "format": "JPEG",
    "file_size_bytes": 182585,
    "exif_orientation": 1
  }
}

First caption document:
{
  "caption_id": 481005,
  "image_i

In [106]:
import json

# Reload saved documents
with open(IMAGE_DOCUMENTS_PATH, "r", encoding="utf-8") as f:
    saved_image_documents = json.load(f)

with open(CAPTION_DOCUMENTS_PATH, "r", encoding="utf-8") as f:
    saved_caption_documents = json.load(f)

# Verify record counts
assert len(saved_image_documents) == TRAINING_SIZE
assert len(saved_caption_documents) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# Verify unique identifiers
assert len({
    doc["image_id"] for doc in saved_image_documents
}) == TRAINING_SIZE

assert len({
    doc["caption_id"] for doc in saved_caption_documents
}) == TRAINING_SIZE * CAPTIONS_PER_IMAGE

# Verify embedding references
assert all(
    doc["embedding"]["index"] == "clip_image_index"
    and 0 <= doc["embedding"]["row"] < TRAINING_SIZE
    for doc in saved_image_documents
)

assert all(
    doc["embedding"]["index"] == "sbert_caption_index"
    and 0 <= doc["embedding"]["row"] < TRAINING_SIZE * CAPTIONS_PER_IMAGE
    for doc in saved_caption_documents
)

# Verify image-caption relationships
image_ids = {
    doc["image_id"] for doc in saved_image_documents
}

assert all(
    doc["image_id"] in image_ids
    for doc in saved_caption_documents
)

print("Saved document files reopened successfully.")
print("Image documents:", len(saved_image_documents))
print("Caption documents:", len(saved_caption_documents))
print(f"Unique image IDs: {len({doc['image_id'] for doc in saved_image_documents})}")
print(f"Unique caption IDs: {len({doc['caption_id'] for doc in saved_caption_documents})}")
print("Embedding references: PASSED")
print("Image-caption associations: PASSED")
print("\nSaved document validation: PASSED")

Saved document files reopened successfully.
Image documents: 9900
Caption documents: 49500
Unique image IDs: 9900
Unique caption IDs: 49500
Embedding references: PASSED
Image-caption associations: PASSED

Saved document validation: PASSED


In [107]:
import json
import faiss
from datetime import datetime, timezone

# --------------------------------------------------
# 1. Define index and document paths
# --------------------------------------------------

CLIP_INDEX_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "indexes"
    / "clip_image_index"
    / "clip_image_index.faiss"
)

SBERT_INDEX_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "indexes"
    / "sbert_caption_index"
    / "sbert_caption_index.faiss"
)

IMAGE_DOCUMENTS_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "documents"
    / "image_documents.json"
)

CAPTION_DOCUMENTS_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "documents"
    / "caption_documents.json"
)

METADATA_DIR = KNOWLEDGE_BASE_ROOT / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

KNOWLEDGE_BASE_METADATA_PATH = (
    METADATA_DIR / "knowledge_base_metadata.json"
)


# --------------------------------------------------
# 2. Read index properties from saved files
# --------------------------------------------------

clip_index_info = faiss.read_index(str(CLIP_INDEX_PATH))
sbert_index_info = faiss.read_index(str(SBERT_INDEX_PATH))


# --------------------------------------------------
# 3. Build knowledge-base metadata
# --------------------------------------------------

knowledge_base_metadata = {
    "pipeline_configuration": get_pipeline_snapshot(),
    "metadata_version": "1.0",

    "knowledge_base": {
        "name": "MS-COCO Image Captioning Knowledge Base",
        "dataset": "MS-COCO 2017",
        "split": "training",
        "random_seed": 45,
        "image_count": TRAINING_SIZE,
        "caption_count": TRAINING_SIZE * CAPTIONS_PER_IMAGE,
        "captions_per_image": CAPTIONS_PER_IMAGE
    },

    "indexes": {
        "clip_image_index": {
            "purpose": "Image embeddings",
            "embedding_model": "openai/clip-vit-base-patch32",
            "index_type": "IndexFlatIP",
            "vector_count": clip_index_info.ntotal,
            "dimension": clip_index_info.d,
            "metric": "Inner Product",
            "normalized_embeddings": True,
            "equivalent_similarity": "Cosine similarity",
            "dtype": "float32",
            "construction_time_seconds": 0.0126,
            "file": "indexes/clip_image_index/clip_image_index.faiss",
            "file_size_bytes": CLIP_INDEX_PATH.stat().st_size,
            "embedding_file": (
                "training/embeddings/clip_image_embeddings.npy"
            ),
            "mapping_file": (
                "training/embeddings/image_embedding_index.json"
            )
        },

        "sbert_caption_index": {
            "purpose": "Caption embeddings",
            "embedding_model": (
                "sentence-transformers/all-MiniLM-L6-v2"
            ),
            "index_type": "IndexFlatIP",
            "vector_count": sbert_index_info.ntotal,
            "dimension": sbert_index_info.d,
            "metric": "Inner Product",
            "normalized_embeddings": True,
            "equivalent_similarity": "Cosine similarity",
            "dtype": "float32",
            "construction_time_seconds": 0.0578,
            "file": "indexes/sbert_caption_index/sbert_caption_index.faiss",
            "file_size_bytes": SBERT_INDEX_PATH.stat().st_size,
            "embedding_file": (
                "training/embeddings/sbert_caption_embeddings.npy"
            ),
            "mapping_file": (
                "training/embeddings/caption_embedding_index.json"
            )
        }
    },

    "documents": {
        "image_documents": {
            "count": len(saved_image_documents),
            "file": "documents/image_documents.json",
            "file_size_bytes": IMAGE_DOCUMENTS_PATH.stat().st_size
        },

        "caption_documents": {
            "count": len(saved_caption_documents),
            "file": "documents/caption_documents.json",
            "file_size_bytes": CAPTION_DOCUMENTS_PATH.stat().st_size
        }
    },

    "relationships": {
        "image_document_to_clip_embedding": (
            "image_id and embedding.row"
        ),
        "caption_document_to_sbert_embedding": (
            "caption_id and embedding.row"
        ),
        "caption_to_image": (
            "caption_document.image_id references image_document.image_id"
        ),
        "image_to_captions": (
            "image_document.caption_ids contains associated caption IDs"
        )
    },

    "data_exclusion": {
        "validation_included": False,
        "test_included": False,
        "description": (
            f"Only the {TRAINING_SIZE} training images and their "
            f"{TRAINING_SIZE * CAPTIONS_PER_IMAGE} captions "
            "are included in this knowledge base."
        )
    },

    "construction_date_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "construction_status": "completed"
}


# --------------------------------------------------
# 4. Save metadata
# --------------------------------------------------

with open(
    KNOWLEDGE_BASE_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        knowledge_base_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# 5. Verify saved metadata
# --------------------------------------------------

with open(
    KNOWLEDGE_BASE_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_metadata = json.load(f)

assert saved_metadata["indexes"]["clip_image_index"]["vector_count"] == TRAINING_SIZE
assert saved_metadata["indexes"]["clip_image_index"]["dimension"] == 512

assert saved_metadata["indexes"]["sbert_caption_index"]["vector_count"] == TRAINING_SIZE * CAPTIONS_PER_IMAGE
assert saved_metadata["indexes"]["sbert_caption_index"]["dimension"] == 384

assert saved_metadata["documents"]["image_documents"]["count"] == TRAINING_SIZE
assert saved_metadata["documents"]["caption_documents"]["count"] == TRAINING_SIZE * CAPTIONS_PER_IMAGE

assert saved_metadata["data_exclusion"]["validation_included"] is False
assert saved_metadata["data_exclusion"]["test_included"] is False

print("Knowledge-base metadata created successfully.")
print("Metadata file:", KNOWLEDGE_BASE_METADATA_PATH)
print("CLIP vectors:", saved_metadata["indexes"]["clip_image_index"]["vector_count"])
print("SBERT vectors:", saved_metadata["indexes"]["sbert_caption_index"]["vector_count"])
print("Image documents:", saved_metadata["documents"]["image_documents"]["count"])
print("Caption documents:", saved_metadata["documents"]["caption_documents"]["count"])
print("\nKnowledge-base metadata validation: PASSED")
check_pipeline_compatibility(
    saved_metadata["pipeline_configuration"],
    "Embedding metadata",
)


Knowledge-base metadata created successfully.
Metadata file: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\knowledge_base\metadata\knowledge_base_metadata.json
CLIP vectors: 9900
SBERT vectors: 49500
Image documents: 9900
Caption documents: 49500

Knowledge-base metadata validation: PASSED


In [108]:
import json
from datetime import datetime, timezone

# --------------------------------------------------
# 1. Prepare manifest directory
# --------------------------------------------------

MANIFEST_DIR = KNOWLEDGE_BASE_ROOT / "manifest"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = (
    MANIFEST_DIR / "knowledge_base_manifest.json"
)

# --------------------------------------------------
# 2. Define files belonging to the knowledge base
# --------------------------------------------------

manifest_files = [
    {
        "name": "CLIP FAISS index",
        "path": "indexes/clip_image_index/clip_image_index.faiss",
        "type": "FAISS index",
        "records": TRAINING_SIZE
    },
    {
        "name": "SBERT FAISS index",
        "path": "indexes/sbert_caption_index/sbert_caption_index.faiss",
        "type": "FAISS index",
        "records": TRAINING_SIZE * CAPTIONS_PER_IMAGE
    },
    {
        "name": "Image documents",
        "path": "documents/image_documents.json",
        "type": "JSON documents",
        "records": TRAINING_SIZE
    },
    {
        "name": "Caption documents",
        "path": "documents/caption_documents.json",
        "type": "JSON documents",
        "records": TRAINING_SIZE * CAPTIONS_PER_IMAGE
    },
    {
        "name": "Knowledge-base metadata",
        "path": "metadata/knowledge_base_metadata.json",
        "type": "JSON metadata",
        "records": None
    }
]

# --------------------------------------------------
# 3. Collect actual file information
# --------------------------------------------------

file_inventory = []

for item in manifest_files:

    file_path = KNOWLEDGE_BASE_ROOT / item["path"]

    assert file_path.exists(), (
        f"Missing knowledge-base file: {file_path}"
    )

    file_inventory.append({
        "name": item["name"],
        "path": item["path"],
        "type": item["type"],
        "records": item["records"],
        "file_size_bytes": file_path.stat().st_size
    })


# --------------------------------------------------
# 4. Build manifest
# --------------------------------------------------

manifest = {
    "manifest_version": "1.0",

    "knowledge_base": "MS-COCO Image Captioning Knowledge Base",

    "dataset": "MS-COCO 2017",

    "split": "training",

    "statistics": {
        "image_count": TRAINING_SIZE,
        "caption_count": TRAINING_SIZE * CAPTIONS_PER_IMAGE,
        "clip_vector_count": TRAINING_SIZE,
        "sbert_vector_count": TRAINING_SIZE * CAPTIONS_PER_IMAGE,
        "total_indexed_vectors": 59400
    },

    "files": file_inventory,

    "excluded_data": [
        "validation",
        "test"
    ],

    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "status": "completed"
}


# --------------------------------------------------
# 5. Save manifest
# --------------------------------------------------

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# 6. Reload and validate manifest
# --------------------------------------------------

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_manifest = json.load(f)

assert len(saved_manifest["files"]) == 5
assert saved_manifest["statistics"]["total_indexed_vectors"] == 59400

for item in saved_manifest["files"]:
    file_path = KNOWLEDGE_BASE_ROOT / item["path"]

    assert file_path.exists()
    assert file_path.stat().st_size == item["file_size_bytes"]

print("Knowledge-base manifest created successfully.")

print("\nFile inventory:")

for item in saved_manifest["files"]:
    print(
        f"- {item['name']}: "
        f"{item['records']} records, "
        f"{item['file_size_bytes']:,} bytes"
    )

print("\nTotal indexed vectors:",
      saved_manifest["statistics"]["total_indexed_vectors"])

print("\nManifest saved to:", MANIFEST_PATH)

print("\nKnowledge-base manifest validation: PASSED")

Knowledge-base manifest created successfully.

File inventory:
- CLIP FAISS index: 9900 records, 20,275,245 bytes
- SBERT FAISS index: 49500 records, 76,032,045 bytes
- Image documents: 9900 records, 9,188,590 bytes
- Caption documents: 49500 records, 15,681,048 bytes
- Knowledge-base metadata: None records, 2,630 bytes

Total indexed vectors: 59400

Manifest saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\knowledge_base\manifest\knowledge_base_manifest.json

Knowledge-base manifest validation: PASSED
